# Notebook 02 — FanGraphs Data Pull

Source: `fangraphs.com` leaderboards via `pybaseball` (handles Cloudflare), with direct URL fallback  
Auth: None required  
Rate limit: 1.5 s between requests, browser-like User-Agent  

This notebook pulls three datasets for the 2025 season:

1. Hitting leaderboard: advanced stats: wRC+, WAR, wOBA, BABIP, batted-ball profile (GB/FB/LD%)  
2. Pitching leaderboard: FIP, xFIP, SIERA, WAR, GB%, K%, BB%  
3. Park factors: single-year and 5-year rolling factors by team  

It also builds a FanGraphs to MLBAM ID crosswalk using normalized name matching, which is the bridge table for merging FanGraphs data with MLB Stats API and Statcast data downstream.

FanGraphs uses its own `fg_id` (IDfg) internally. The MLBAM ID (`mlbam_id`) is the universal join key across all other sources.

In [112]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from scrapers.fangraphs import (
    get_fg_hitting, get_fg_pitching, get_park_factors, build_fg_mlbam_crosswalk
)
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

## 1. FanGraphs Hitting Leaderboard

**Key columns:**
- `wRC+`: Weighted Runs Created Plus. Park- and league-adjusted; 100 = average. The best single "how good is this hitter" number.
- `WAR`: Wins Above Replacement (FanGraphs version, fWAR).
- `wOBA`: Weighted On-Base Average. Run-value-weighted version of OBP. This fills the gap left by the MLB Stats API (which doesn't provide wOBA).
- `BABIP`: Batting Average on Balls In Play. Useful for spotting luck-driven outliers.
- `GB%` / `FB%` / `LD%`: Batted ball profile. Ground ball, fly ball, and line drive rates.

In [114]:
fg_hitting = get_fg_hitting(2025)
print(f"Shape: {fg_hitting.shape}")
print(f"\nDtypes:\n{fg_hitting.dtypes}")
print(f"\nNull rates:\n{fg_hitting.isnull().mean().round(4)}")
print(f"\nSample (10 rows):")
fg_hitting.head(10)

Shape: (537, 10)

Dtypes:
fg_id         int64
name         object
team         object
wRC_plus      int64
WAR         float64
wOBA        float64
BABIP       float64
GB_pct      float64
FB_pct      float64
LD_pct      float64
dtype: object

Null rates:
fg_id       0.0
name        0.0
team        0.0
wRC_plus    0.0
WAR         0.0
wOBA        0.0
BABIP       0.0
GB_pct      0.0
FB_pct      0.0
LD_pct      0.0
dtype: float64

Sample (10 rows):


,fg_id,name,team,wRC_plus,WAR,wOBA,BABIP,GB_pct,FB_pct,LD_pct
0,15640,Aaron Judge,NYY,204,10.1,0.463,0.376,0.340,0.464,0.196
1,21534,Cal Raleigh,SEA,161,9.1,0.392,0.248,0.251,0.577,0.173
2,25764,Bobby Witt Jr.,KCR,130,8.0,0.360,0.334,0.370,0.428,0.202
3,19755,Shohei Ohtani,LAD,172,7.5,0.418,0.315,0.392,0.439,0.169
4,22799,Geraldo Perdomo,ARI,138,7.1,0.370,0.303,0.375,0.392,0.233
5,16252,Trea Turner,PHI,125,6.7,0.352,0.350,0.479,0.329,0.192
6,25878,Corbin Carroll,ARI,139,6.5,0.371,0.299,0.347,0.453,0.200
7,13510,Jose Ramirez,CLE,133,6.3,0.360,0.279,0.334,0.464,0.202
8,12916,Francisco Lindor,NYM,129,6.3,0.350,0.288,0.377,0.402,0.221
9,19709,Fernando Tatis Jr.,SDP,131,6.1,0.353,0.303,0.489,0.339,0.172


## 2. FanGraphs Pitching Leaderboard

**Key columns:**
- `FIP`: Fielding Independent Pitching. Estimates ERA from K, BB, HR, HBP only. Strips out defense.
- `xFIP`: Expected FIP. Same as FIP but normalizes HR/FB rate to the league average (~10%). More predictive than FIP for future performance.
- `SIERA`: Skill-Interactive ERA. The most sophisticated ERA estimator — accounts for batted ball types and their interaction with K/BB rates.
- `WAR`: Pitcher fWAR.
- `GB%`: Ground ball rate. High-GB pitchers suppress HR and induce double plays.
- `K%` / `BB%`: Strikeout and walk rates (as proportions of PA faced).

In [116]:
fg_pitching = get_fg_pitching(2025)
print(f"Shape: {fg_pitching.shape}")
print(f"\nDtypes:\n{fg_pitching.dtypes}")
print(f"\nNull rates:\n{fg_pitching.isnull().mean().round(4)}")
print(f"\nSample (10 rows):")
fg_pitching.head(10)

Shape: (657, 10)

Dtypes:
fg_id       int64
name       object
team       object
FIP       float64
xFIP      float64
SIERA     float64
WAR       float64
GB_pct    float64
K_pct     float64
BB_pct    float64
dtype: object

Null rates:
fg_id     0.0
name      0.0
team      0.0
FIP       0.0
xFIP      0.0
SIERA     0.0
WAR       0.0
GB_pct    0.0
K_pct     0.0
BB_pct    0.0
dtype: float64

Sample (10 rows):


,fg_id,name,team,FIP,xFIP,SIERA,WAR,GB_pct,K_pct,BB_pct
0,22267,Tarik Skubal,DET,2.45,2.66,2.71,6.6,0.410,0.322,0.044
1,33677,Paul Skenes,PIT,2.36,3.03,3.10,6.5,0.443,0.295,0.057
2,20778,Cristopher Sanchez,PHI,2.55,2.77,3.02,6.4,0.583,0.263,0.055
3,27463,Garrett Crochet,BOS,2.89,2.64,2.86,5.8,0.483,0.313,0.057
4,17995,Logan Webb,SFG,2.60,2.78,3.14,5.5,0.532,0.262,0.054
5,19959,Jesus Luzardo,PHI,2.90,3.25,3.40,5.3,0.428,0.285,0.075
6,33825,Yoshinobu Yamamoto,LAD,2.94,3.05,3.32,5.0,0.528,0.294,0.086
7,13743,Max Fried,NYY,3.07,3.41,3.60,4.8,0.524,0.236,0.064
8,25880,Hunter Brown,HOU,3.14,3.19,3.39,4.6,0.481,0.283,0.078
9,14107,Kevin Gausman,TOR,3.41,3.77,3.79,4.1,0.367,0.244,0.065


## 3. Park Factors

Park factors quantify how much a team's home ballpark inflates or suppresses run scoring relative to a neutral park.

- `park_factor_basic`: single-season (1yr) factor. 100 = neutral. >100 = hitter-friendly. <100 = pitcher-friendly.
- `park_factor_5yr`: 5-year rolling average. More stable and preferred for projections since single-year factors are noisy.

In [118]:
park_factors = get_park_factors(2025)
print(f"Shape: {park_factors.shape}")
print(f"\nDtypes:\n{park_factors.dtypes}")
print(f"\nNull rates:\n{park_factors.isnull().mean().round(4)}")
print(f"\nAll 30 teams:")
park_factors

Shape: (30, 3)

Dtypes:
team                 object
park_factor_basic     int64
park_factor_5yr       int64
dtype: object

Null rates:
team                 0.0
park_factor_basic    0.0
park_factor_5yr      0.0
dtype: float64

All 30 teams:


,team,park_factor_basic,park_factor_5yr
0,Angels,101,101
1,Orioles,103,99
2,Red Sox,100,104
3,White Sox,99,100
4,Guardians,100,99
5,Tigers,104,100
6,Royals,95,103
7,Twins,106,101
8,Yankees,96,99
9,Athletics,105,103


## 4. FanGraphs → MLBAM ID Crosswalk

The crosswalk joins FanGraphs data to the MLBAM player universe using a two-pass name matching strategy:

1. Pass 1 — Exact normalized name: Strip accents (José→Jose, Ramón→Ramon), remove suffixes (Jr., Sr., II), lowercase, remove punctuation. Match on full name.
2. Pass 2 — Last name + team: For any remaining unmatched, try matching on last name + team abbreviation to handle first-name variants.

Potential systematic mismatches:
- Mid-season trades: FanGraphs assigns traded players to `"- - -"` (multiple teams). The MLB API only shows the current team, so pass-2 (last name + team) won't match these. Pass-1 (name only) handles most of them.
- Accent characters: FanGraphs strips accents; the MLB API preserves them. Our `unicodedata` normalization handles this.
- Name suffixes: "Ronald Acuña Jr." vs "Ronald Acuna" — suffix removal handles this.

In [120]:
# Load the MLBAM players roster from Step 3
mlbam_players = pd.read_parquet(os.path.join("..", "data", "raw_mlb_api_players.parquet"))

# Build crosswalk from hitting leaderboard
xwalk_hit = build_fg_mlbam_crosswalk(fg_hitting, mlbam_players)

total = len(xwalk_hit)
matched = (xwalk_hit["match_status"] == "matched").sum()
unmatched = total - matched

print(f"=== Hitting Crosswalk ===")
print(f"Total FG hitters: {total}")
print(f"Matched to MLBAM: {matched} ({matched/total*100:.1f}%)")
print(f"Unmatched:         {unmatched} ({unmatched/total*100:.1f}%)")

if unmatched > 0:
    print(f"\nUnmatched rows:")
    print(xwalk_hit[xwalk_hit["match_status"] == "unmatched"][["fg_id", "name", "team"]].to_string(index=False))

=== Hitting Crosswalk ===
Total FG hitters: 537
Matched to MLBAM: 537 (100.0%)
Unmatched:         0 (0.0%)


In [121]:
# Build crosswalk from pitching leaderboard too
xwalk_pit = build_fg_mlbam_crosswalk(fg_pitching, mlbam_players)

total_p = len(xwalk_pit)
matched_p = (xwalk_pit["match_status"] == "matched").sum()
unmatched_p = total_p - matched_p

print(f"=== Pitching Crosswalk ===")
print(f"Total FG pitchers: {total_p}")
print(f"Matched to MLBAM:  {matched_p} ({matched_p/total_p*100:.1f}%)")
print(f"Unmatched:          {unmatched_p} ({unmatched_p/total_p*100:.1f}%)")

if unmatched_p > 0:
    print(f"\nUnmatched rows:")
    print(xwalk_pit[xwalk_pit["match_status"] == "unmatched"][["fg_id", "name", "team"]].to_string(index=False))

=== Pitching Crosswalk ===
Total FG pitchers: 657
Matched to MLBAM:  653 (99.4%)
Unmatched:          4 (0.6%)

Unmatched rows:
 fg_id               name  team
 32095 Cameron Schlittler   NYY
 27691      Louie Varland - - -
 27646      Luis L. Ortiz   CLE
 31468    Jackson Perkins   ATH


In [122]:
# Combine both crosswalks into a single unified table (dedup by fg_id)
xwalk_combined = pd.concat([xwalk_hit, xwalk_pit], ignore_index=True)
xwalk_combined = xwalk_combined.drop_duplicates(subset=["fg_id"], keep="first")

total_c = len(xwalk_combined)
matched_c = (xwalk_combined["match_status"] == "matched").sum()

print(f"=== Combined Crosswalk ===")
print(f"Total unique fg_ids: {total_c}")
print(f"Matched: {matched_c} ({matched_c/total_c*100:.1f}%)")
print(f"Unmatched: {total_c - matched_c}")

# Flag mid-season trade players (team = "- - -")
trade_players = xwalk_combined[xwalk_combined["team"] == "- - -"]
if len(trade_players) > 0:
    print(f"\n=== Players on multiple teams (traded mid-season): {len(trade_players)} ===")
    print(trade_players[["fg_id", "mlbam_id", "name", "team", "match_status"]].to_string(index=False))

=== Combined Crosswalk ===
Total unique fg_ids: 1193
Matched: 1189 (99.7%)
Unmatched: 4

=== Players on multiple teams (traded mid-season): 171 ===
 fg_id  mlbam_id                 name  team match_status
 12552    553993       Eugenio Suarez - - -      matched
 17350    646240        Rafael Devers - - -      matched
 18030    664056       Harrison Bader - - -      matched
 18839    647304          Josh Naylor - - -      matched
 17128    657656       Ramon Laureano - - -      matched
 16442    656811         Ryan O'Hearn - - -      matched
 14162    621043        Carlos Correa - - -      matched
 14854    573262     Mike Yastrzemski - - -      matched
 23401    676609       Jose Caballero - - -      matched
 15112    641857         Ryan McMahon - - -      matched
 20572    665019         Kody Clemens - - -      matched
 18577    663647       Ke'Bryan Hayes - - -      matched
 16535    643376         Danny Jansen - - -      matched
 17929    656775       Cedric Mullins - - -      match

## 5. Null Rate Summary (all DataFrames)

In [124]:
null_summary = pd.DataFrame({
    "fg_hitting": fg_hitting.isnull().mean().round(4),
    "fg_pitching": fg_pitching.isnull().mean().round(4),
    "park_factors": park_factors.isnull().mean().round(4),
    "crosswalk": xwalk_combined.isnull().mean().round(4),
}).fillna("—")
print("=== Null Rate Summary ===")
null_summary

=== Null Rate Summary ===


,fg_hitting,fg_pitching,park_factors,crosswalk
BABIP,0.0,—,—,—
BB_pct,—,0.0,—,—
FB_pct,0.0,—,—,—
FIP,—,0.0,—,—
GB_pct,0.0,0.0,—,—
K_pct,—,0.0,—,—
LD_pct,0.0,—,—,—
SIERA,—,0.0,—,—
WAR,0.0,0.0,—,—
fg_id,0.0,0.0,—,0.0


## 6. Data Quality Notes

1. wOBA from FanGraphs fills the MLB API gap — the MLB Stats API doesn't provide wOBA, but FanGraphs does. We'll use the crosswalk to map it onto the MLBAM-keyed hitting data downstream.

3. FanGraphs team abbreviations — FG uses 2-3 letter abbreviations (NYY, LAD, etc.) while the MLB API uses full team names. The crosswalk bridges this gap through player-level matching rather than team-level matching.

4. Traded players — FanGraphs lists players who were traded mid-season under team `"- - -"`. These still match via Pass 1 (name-only matching).

## 7. Save to Parquet

In [127]:
data_dir = os.path.join("..", "data")
os.makedirs(data_dir, exist_ok=True)

fg_hitting.to_parquet(os.path.join(data_dir, "raw_fangraphs_hitting.parquet"), index=False)
fg_pitching.to_parquet(os.path.join(data_dir, "raw_fangraphs_pitching.parquet"), index=False)
park_factors.to_parquet(os.path.join(data_dir, "park_factors.parquet"), index=False)
xwalk_combined.to_parquet(os.path.join(data_dir, "fg_mlbam_crosswalk.parquet"), index=False)

print("Saved:")
for f in ["raw_fangraphs_hitting.parquet", "raw_fangraphs_pitching.parquet",
          "park_factors.parquet", "fg_mlbam_crosswalk.parquet"]:
    path = os.path.join(data_dir, f)
    size_kb = os.path.getsize(path) / 1024
    print(f"  {f} — {size_kb:.1f} KB")

Saved:
  raw_fangraphs_hitting.parquet — 27.6 KB
  raw_fangraphs_pitching.parquet — 30.9 KB
  park_factors.parquet — 3.0 KB
  fg_mlbam_crosswalk.parquet — 32.8 KB
